# EasyVisa – Visa Approval Prediction Using Machine Learning

## Project Overview
This notebook implements a machine learning-based solution to predict visa approval status and identify key factors influencing certification decisions for the Office of Foreign Labor Certification (OFLC).

# EasyVisa Project: Visa Approval Prediction

## 1. Exploratory Data Analysis (EDA)

### Problem Definition

The Immigration and Nationality Act (INA) of the US permits foreign workers to come to the United States to work on either a temporary or permanent basis. The process of reviewing every case is becoming a tedious task as the number of applicants is increasing every year.

The objective is to develop a Machine Learning based solution that can help the Office of Foreign Labor Certification (OFLC) in shortlisting the candidates having higher chances of VISA approval. As a data scientist at EasyVisa, the goal is to:
1. Facilitate the process of visa approvals.
2. Recommend a suitable profile for the applicants for whom the visa should be certified or denied based on the drivers that significantly influence the case status.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 8]

# Load the dataset
data = pd.read_csv('EasyVisa.csv')

# Display the first few rows
data.head()

### Observations after loading data:
- The dataset contains multiple features related to both the employee (education, experience, etc.) and the employer (number of employees, year established, etc.).
- The target variable is `case_status`, which indicates if the visa was 'Certified' or 'Denied'.
- Initial inspection shows categorical variables like `continent`, `education_of_employee`, and `unit_of_wage` which will require encoding later.

In [ ]:
# Check data types and non-null counts
print("--- Data Info ---")
data.info()

print("\n--- Missing Values ---")
print(data.isnull().sum())

print("\n--- Duplicates ---")
print(data.duplicated().sum())

### Observations on Data Integrity:
- The dataset consists of 25,480 rows and 12 columns.
- There are no missing values in any of the columns.
- There are no duplicate rows in the dataset.
- Column `case_id` is a unique identifier and does not contribute to the model's predictive power; it should be dropped during pre-processing.

In [ ]:
# Statistical summary of numerical features
data.describe().T

### Observations on Statistical Summary:
- **no_of_employees**: The minimum value is -26, which is logically incorrect as the number of employees cannot be negative. This indicates noise in the data that needs treatment.
- **yr_of_estab**: Ranges from 1800 to 2016. This variable can be used to calculate the age of the company.
- **prevailing_wage**: Shows a wide range (from ~2 to ~319,210). The units differ (Hourly, Weekly, Monthly, Yearly), so these must be normalized to a common scale for meaningful analysis.

### Univariate Analysis

In [ ]:
# Function to create histograms and boxplots for numerical columns
def histogram_boxplot(data, feature, figsize=(12, 7), kde=False, bins=None):
    f2, (ax_box2, ax_hist2) = plt.subplots(
        nrows=2, 
        sharex=True, 
        gridspec_kw={"height_ratios": (0.25, 0.75)}, 
        figsize=figsize
    )
    sns.boxplot(data=data, x=feature, ax=ax_box2, showmeans=True, color="violet")
    sns.histplot(data=data, x=feature, kde=kde, ax=ax_hist2, bins=bins, color="teal") if bins else sns.histplot(
        data=data, x=feature, kde=kde, ax=ax_hist2, color="teal"
    )
    ax_hist2.axvline(data[feature].mean(), color="green", linestyle="--")
    ax_hist2.axvline(data[feature].median(), color="black", linestyle="-")
    plt.show()

for col in ['no_of_employees', 'yr_of_estab', 'prevailing_wage']:
    histogram_boxplot(data, col, kde=True)

### Observations on Numerical Distributions:
- **no_of_employees**: Highly right-skewed with significant outliers. Most companies have a relatively small number of employees.
- **yr_of_estab**: Most companies were established after 1950, though some date back to 1800.
- **prevailing_wage**: The distribution is heavily right-skewed with many outliers on the higher end. The multimodal nature likely stems from the different units of wage (Hourly vs Yearly).

In [ ]:
# Function for labeled countplots
def labeled_countplot(data, feature, perc=False, n=None):
    plt.figure(figsize=(12, 6))
    ax = sns.countplot(data=data, x=feature, palette="viridis", order=data[feature].value_counts().index[:n])
    for p in ax.patches:
        if perc:
            label = "{:.1f}%".format(100 * p.get_height() / len(data))
        else:
            label = p.get_height()
        ax.annotate(label, (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='center', 
                    fontsize=12, color='black', xytext=(0, 5), textcoords='offset points')
    plt.xticks(rotation=45)
    plt.show()

cat_cols = ['continent', 'education_of_employee', 'has_job_experience', 
            'requires_job_training', 'region_of_employment', 'unit_of_wage', 
            'full_time_position', 'case_status']

for col in cat_cols:
    labeled_countplot(data, col, perc=True)

### Observations on Categorical Distributions:
- **continent**: Asia has the highest representation in visa applications (~66%).
- **education_of_employee**: Most applicants have a Bachelor's or Master's degree.
- **case_status**: Approximately 66.8% of cases are 'Certified', indicating a class imbalance that we may need to address (Oversampling/Undersampling).
- **unit_of_wage**: The vast majority (~90%) of wages are recorded on a 'Year' basis.

### Bivariate Analysis

In [ ]:
# Function for stacked bar plots to compare features against case_status
def stacked_barplot(data, predictor, target):
    count = data.groupby([predictor, target]).size().unstack()
    perc = (count.T / count.sum(axis=1)).T
    perc.plot(kind="bar", stacked=True, figsize=(12, 6), color=['salmon', 'skyblue'])
    plt.ylabel("Percentage")
    plt.legend(title=target, loc='upper right')
    plt.show()

for col in cat_cols[:-1]: # Exclude target itself
    stacked_barplot(data, col, 'case_status')

### Observations on Bivariate Analysis (Categorical vs Target):
- **education_of_employee**: Applicants with Doctorate and Master's degrees have significantly higher certification rates compared to those with only High School education.
- **has_job_experience**: Applicants with prior job experience are much more likely to have their visa certified.
- **continent**: Europe and Africa show relatively higher certification rates compared to other continents.

In [ ]:
# Numerical vs Target using boxplots
num_cols = ['no_of_employees', 'yr_of_estab', 'prevailing_wage']
plt.figure(figsize=(15, 10))
for i, col in enumerate(num_cols):
    plt.subplot(2, 2, i+1)
    sns.boxplot(data=data, x='case_status', y=col, palette="magma")
plt.tight_layout()
plt.show()

### Observations on Bivariate Analysis (Numerical vs Target):
- **prevailing_wage**: Certified cases generally have a slightly higher median prevailing wage compared to denied cases, though the overlap is significant.
- **no_of_employees**: There isn't a very clear distinction in the distribution of company size between certified and denied cases, although extremely large companies seem to have high certification rates.